# AIS Exploration for ML and STS

Этот ноутбук работает с новым пайплайном и тремя таблицами:
- `ais_observations`: сырые AIS-наблюдения;
- `vessel_zone_events`: эпизоды нахождения судна в целевых зонах;
- `sts_candidates`: кандидаты на STS по overlap, дистанции, скорости и осадке.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

from lng_ml_research.ais_pipeline import AISDatasetBuilder

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


## 1. Конфигурация источника

Ноутбук работает с `json/jsonl` raw-источниками из `data/raw`. Можно анализировать либо всю папку целиком, либо конкретный файл вроде `ais_observations.jsonl` или `lng_tracker_dataset.json`.


In [ ]:
RAW_DIR = ROOT / "data" / "raw"
raw_files = sorted(path.name for path in RAW_DIR.glob("*.json*"))
print("Available JSON/JSONL raw files:")
for file_name in raw_files:
    print(f"- {file_name}")


In [ ]:
INPUT_SOURCE = RAW_DIR
# INPUT_SOURCE = RAW_DIR / "ais_observations.jsonl"
# INPUT_SOURCE = RAW_DIR / "lng_tracker_dataset.json"
# INPUT_SOURCE = "https://tankermap.com/api/vessels/live"

builder = AISDatasetBuilder(event_gap_minutes=360, distance_tolerance_minutes=30)


## 2. Построение датасетов

Пайплайн читает совместимые `json/jsonl` файлы из `raw`, строит три таблицы и возвращает краткую сводку по данным.


In [ ]:
observations_df, events_df, sts_df, loitering_df, congestion_df, summary = builder.build_all(INPUT_SOURCE)

print(summary)
print(f"observations: {len(observations_df)}")
print(f"events: {len(events_df)}")
print(f"sts candidates: {len(sts_df)}")
print(f"loitering candidates: {len(loitering_df)}")
print(f"congestion windows: {len(congestion_df)}")


## 3. AIS observations

Смотрим структуру сырого AIS-слоя после нормализации типов и определения зоны.


In [ ]:
observations_df.head(10)


In [ ]:
display(observations_df.info())
display(observations_df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))
display(observations_df["zone"].value_counts(dropna=False).to_frame("count"))


In [ ]:
zone_plot_df = observations_df.assign(zone_label=observations_df["zone"].fillna("outside_target_zones"))
zone_order = zone_plot_df["zone_label"].value_counts().index

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.countplot(
    data=zone_plot_df,
    y="zone_label",
    hue="zone_label",
    order=zone_order,
    hue_order=zone_order,
    ax=axes[0],
    palette="Blues_r",
    legend=False,
)
axes[0].set_title("Observations by Zone")
axes[0].set_xlabel("Count")
axes[0].set_ylabel("Zone")

speed_plot_df = observations_df.dropna(subset=["speed_knots"])
sns.histplot(speed_plot_df["speed_knots"], bins=30, kde=True, ax=axes[1], color="#2a9d8f")
axes[1].set_title("Speed Distribution")
axes[1].set_xlabel("Speed (knots)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()


## 4. Zone events

Это event-level слой: один эпизод нахождения судна в зоне подряд.


In [ ]:
events_df.head(10)


In [ ]:
event_summary = (
    events_df.groupby("zone", dropna=False)
    .agg(
        events=("zone", "size"),
        unique_vessels=("mmsi", "nunique"),
        avg_duration_hours=("duration_hours", "mean"),
        median_duration_hours=("duration_hours", "median"),
        avg_speed_knots=("avg_speed_knots", "mean"),
        avg_draught_change=("draught_change_meters", "mean"),
    )
    .sort_values("events", ascending=False)
)

event_summary


In [ ]:
zone_order = events_df["zone"].value_counts().index
status_order = events_df["status"].value_counts().index

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.countplot(
    data=events_df,
    y="zone",
    hue="zone",
    order=zone_order,
    hue_order=zone_order,
    ax=axes[0],
    palette="crest",
    legend=False,
)
axes[0].set_title("Zone Events by Zone")
axes[0].set_xlabel("Count")
axes[0].set_ylabel("Zone")

sns.countplot(
    data=events_df,
    x="status",
    hue="status",
    order=status_order,
    hue_order=status_order,
    ax=axes[1],
    palette="Set2",
    legend=False,
)
axes[1].set_title("Event Status Distribution")
axes[1].set_xlabel("Status")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()


In [ ]:
zone_order = events_df["zone"].value_counts().index

plt.figure(figsize=(14, 6))
sns.boxplot(
    data=events_df,
    x="zone",
    y="duration_hours",
    hue="zone",
    order=zone_order,
    hue_order=zone_order,
    palette="pastel",
    legend=False,
)
plt.title("Event Duration by Zone")
plt.xlabel("Zone")
plt.ylabel("Duration (hours)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 5. STS candidates

В новом пайплайне `sts_candidates` считаются уже не только по overlap во времени, но и с учётом дистанции между судами, средней скорости и изменения осадки.


In [ ]:
sts_df.head(15)


In [ ]:
sts_df.describe(include="all").transpose()


In [ ]:
if not sts_df.empty:
    plot_sts = sts_df.head(10).copy().iloc[::-1]
    plot_sts["pair"] = plot_sts["vessel_a_name"] + " <> " + plot_sts["vessel_b_name"]

    plt.figure(figsize=(14, 8))
    plt.barh(plot_sts["pair"], plot_sts["sts_score"], color="#457b9d")
    plt.title("Top STS Candidate Pairs")
    plt.xlabel("STS Score")
    plt.ylabel("Vessel Pair")
    plt.tight_layout()
    plt.show()
else:
    print("No STS candidates were found for the current data and thresholds.")


In [ ]:
if not sts_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    sns.scatterplot(
        data=sts_df,
        x="min_distance_nm",
        y="sts_score",
        hue="zone",
        ax=axes[0],
        palette="Set1",
    )
    axes[0].set_title("STS Score vs Min Distance")
    axes[0].set_xlabel("Min distance (nautical miles)")
    axes[0].set_ylabel("STS score")

    sts_df_plot = sts_df.copy()
    sts_df_plot["pair_avg_speed"] = (sts_df_plot["vessel_a_avg_speed"] + sts_df_plot["vessel_b_avg_speed"]) / 2
    sns.scatterplot(
        data=sts_df_plot,
        x="pair_avg_speed",
        y="sts_score",
        hue="zone",
        ax=axes[1],
        palette="Set1",
    )
    axes[1].set_title("STS Score vs Pair Average Speed")
    axes[1].set_xlabel("Average speed (knots)")
    axes[1].set_ylabel("STS score")

    plt.tight_layout()
    plt.show()
else:
    print("STS scatter plots skipped because sts_df is empty.")


## 6. Loitering candidates

Слой `loitering_candidates` ранжирует события, где судно дольше обычного находится в зоне и при этом движется медленно.


In [ ]:
loitering_df.head(15)


In [ ]:
if not loitering_df.empty:
    plot_loitering = loitering_df.head(10).copy().iloc[::-1]

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    sns.barplot(data=plot_loitering, x="loitering_score", y="name", hue="zone", dodge=False, ax=axes[0], palette="rocket")
    axes[0].set_title("Top Loitering Candidates")
    axes[0].set_xlabel("Loitering score")
    axes[0].set_ylabel("Vessel")
    axes[0].legend(loc="lower right")

    sns.scatterplot(data=loitering_df, x="duration_hours", y="avg_speed_knots", size="loitering_score", hue="zone", ax=axes[1], palette="Set2")
    axes[1].set_title("Duration vs Speed for Loitering Analysis")
    axes[1].set_xlabel("Duration (hours)")
    axes[1].set_ylabel("Average speed (knots)")

    plt.tight_layout()
    plt.show()
else:
    print("No loitering candidates were found for the current data.")


## 7. Zone congestion

Слой `zone_congestion` показывает, в какие часы зона была наиболее загружена по числу судов, наблюдений и замедлению трафика.


In [ ]:
congestion_df.head(15)


In [ ]:
if not congestion_df.empty:
    plot_congestion = congestion_df.head(12).copy().iloc[::-1]
    plot_congestion["window"] = plot_congestion["zone"] + " | " + plot_congestion["observed_hour"].astype(str)

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    sns.barplot(data=plot_congestion, x="congestion_score", y="window", hue="zone", dodge=False, ax=axes[0], palette="mako")
    axes[0].set_title("Top Congestion Windows")
    axes[0].set_xlabel("Congestion score")
    axes[0].set_ylabel("Zone | Hour")
    axes[0].legend(loc="lower right")

    sns.scatterplot(data=congestion_df, x="unique_vessels", y="avg_speed_knots", size="congestion_score", hue="zone", ax=axes[1], palette="tab10")
    axes[1].set_title("Congestion: Vessel Count vs Average Speed")
    axes[1].set_xlabel("Unique vessels")
    axes[1].set_ylabel("Average speed (knots)")

    plt.tight_layout()
    plt.show()
else:
    print("No congestion windows were found for the current data.")


## 6. Сохранение результатов

Если нужно, можно сохранить построенные таблицы в `data/processed`. Ниже запрашивается `parquet`, но если в окружении нет `pyarrow` или `fastparquet`, код автоматически сохранит результаты в CSV.


In [ ]:
OUTPUT_DIR = ROOT / "data" / "processed"
paths = builder.write_outputs(observations_df, events_df, sts_df, loitering_df, congestion_df, OUTPUT_DIR, output_format="parquet")
paths


## 7. Что можно улучшить дальше

- копить исторические snapshots по расписанию, а не анализировать один вызов API;
- заменить bounding boxes зон на полигоны;
- добавить anomaly detection уже на `vessel_zone_events`;
- учитывать course variability, low-speed ratio и near-stationary ratio прямо в `sts_score`.
